In [1]:
import pandas as pd

df = pd.read_csv("../data/cicids2017/friday.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head())

(547557, 89)
['Src IP dec', 'Src Port', 'Dst IP dec', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag

In [2]:
print(df['Label'].value_counts())

Label
BENIGN                288544
Portscan              159066
DDoS                   95144
Botnet - Attempted      4067
Botnet                   736
Name: count, dtype: int64


In [3]:
# calculating the infinity/NaN Count
import numpy as np
print(df.replace([np.inf, -np.inf], np.nan).isna().sum().sum())

0


In [4]:
# double checking for infinities in the flow Byte's and Flow packet's columns
print(df[['Flow Bytes/s', 'Flow Packets/s']].describe())
print(df[['Flow Bytes/s', 'Flow Packets/s']].isin([np.inf, -np.inf]).sum())

       Flow Bytes/s  Flow Packets/s
count  5.475570e+05    5.475570e+05
mean   3.371764e+05    2.421394e+04
std    3.324236e+06    9.454686e+04
min    0.000000e+00    2.500746e-02
25%    0.000000e+00    3.002328e+00
50%    1.226604e+03    1.071438e+02
75%    6.368100e+03    2.898551e+04
max    1.781765e+08    2.000000e+06
Flow Bytes/s      0
Flow Packets/s    0
dtype: int64


In [5]:
# label mapping
df['label_binary'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print(df['label_binary'].value_counts())

label_binary
0    288544
1    259013
Name: count, dtype: int64


In [6]:
exclude_cols = ['Label', 'label_binary', 'Attempted Category', 'Timestamp', 'Src IP dec', 'Dst IP dec']
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols]
y = df['label_binary']

print(X.shape)
print(X.dtypes.value_counts())

(547557, 84)
int64      59
float64    25
Name: count, dtype: int64


In [7]:
# train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(438045, 84) (109512, 84)
label_binary
0    230835
1    207210
Name: count, dtype: int64
label_binary
0    57709
1    51803
Name: count, dtype: int64


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     57709
           1       1.00      1.00      1.00     51803

    accuracy                           1.00    109512
   macro avg       1.00      1.00      1.00    109512
weighted avg       1.00      1.00      1.00    109512

[[57709     0]
 [    4 51799]]


In [9]:
# checking feature importances
import pandas as pd
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))

Total Length of Fwd Packet    0.147828
Fwd Packet Length Mean        0.115787
Subflow Fwd Bytes             0.089245
Fwd Packet Length Max         0.081029
Fwd Segment Size Avg          0.079543
RST Flag Count                0.063815
SYN Flag Count                0.037858
Total TCP Flow Time           0.029978
Packet Length Min             0.028330
ACK Flag Count                0.027158
Protocol                      0.018945
Fwd Packet Length Std         0.016658
Fwd Seg Size Min              0.014718
Fwd Packet Length Min         0.014192
PSH Flag Count                0.014127
dtype: float64


In [ ]:
# cross validation
from sklearn.model_selection import cross_val_score
scores = cross_val_score(rf, X, y, cv=5, scoring='f1')
print(scores)
print(scores.mean(), scores.std())